In [1]:
import os
import asyncio
from dotenv import find_dotenv, load_dotenv
from openai import AsyncOpenAI
from ragas_examples.improve_rag.rag import RAG, BM25Retriever

# Set up OpenAI client
_ = load_dotenv(find_dotenv())
openai_client = AsyncOpenAI()

/workspaces/123/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
import types

vertexai_mod = types.ModuleType("langchain_community.chat_models.vertexai")

class ChatVertexAI:
    pass

vertexai_mod.ChatVertexAI = ChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = vertexai_mod

In [3]:
import sys
from pathlib import Path

# 如果 notebook 在 Naive RAG/week2 运行
sys.path.append(str(Path("../week1").resolve()))

from app.rag_chain import create_rag_chain

my_rag = create_rag_chain()

2026-05-29 18:09:17.006649374 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [4]:
from pathlib import Path
import pandas as pd
from ragas import Dataset

def create_my_ragas_dataset() -> Dataset:
    dataset_path = Path("datasets/my_rag_eval.csv")
    df = pd.read_csv(dataset_path)

    dataset = Dataset(
        name="my_rag_eval",
        backend="local/csv",
        root_dir="."
    )

    for _, row in df.iterrows():
        dataset.append({
            "question": row["question"],
            "expected_answer": row["expected_answer"]
        })

    dataset.save()
    return dataset

dataset = create_my_ragas_dataset()

In [5]:
# examples/ragas_examples/improve_rag/evals.py
from ragas.metrics import DiscreteMetric

# Define correctness metric
correctness_metric = DiscreteMetric(
    name="correctness",
    prompt="""Compare the model response to the expected answer and determine if it's correct.

Consider the response correct if it:
1. Contains the key information from the expected answer
2. Is factually accurate based on the provided context
3. Adequately addresses the question asked

Return 'pass' if the response is correct, 'fail' if it's incorrect.

Question: {question}
Expected Answer: {expected_answer}
Model Response: {response}

Evaluation:""",
    allowed_values=["pass", "fail"],
)

In [6]:
# examples/ragas_examples/improve_rag/evals.py
import asyncio
from typing import Dict, Any
from ragas import experiment

@experiment()
async def evaluate_rag(row: Dict[str, Any], rag: RAG, llm) -> Dict[str, Any]:
    """
    Run RAG evaluation on a single row.

    Args:
        row: Dictionary containing question and expected_answer
        rag: Pre-initialized RAG instance
        llm: Pre-initialized LLM client for evaluation

    Returns:
        Dictionary with evaluation results
    """
    question = row["question"]

    rag_response = await rag.ainvoke({
        "input": question,
        "chat_history": []
    })

    model_response = rag_response.get("answer", "")
    docs = rag_response.get("context", [])

    score = await correctness_metric.ascore(
        question=question,
        expected_answer=row["expected_answer"],
        response=model_response,
        llm=llm
    )

    result = {
        **row,
        "model_response": model_response,
        "correctness_score": score.value,
        "correctness_reason": score.reason,
        "retrieved_documents": [
            doc.page_content[:200] + "..." for doc in docs
        ],
    }

    return result 

In [7]:
# Import required components
import asyncio
from ragas.llms import llm_factory
from datetime import datetime
from ragas_examples.improve_rag.rag import RAG, BM25Retriever

async def run_evaluation():
    load_dotenv(find_dotenv())

    openai_client = AsyncOpenAI()

    rag = create_rag_chain()

    llm = llm_factory(
        "gpt-4o-mini",
        client=openai_client
    )

    exp_name = f"{datetime.now().strftime('%Y%m%d-%H%M%S')}_myrag"

    results = await evaluate_rag.arun(
        dataset,
        name=exp_name,
        rag=rag,
        llm=llm
    )

    pass_count = sum(
        1 for result in results
        if result.get("correctness_score") == "pass"
    )
    total_count = len(results)

    print(f"Results: {pass_count}/{total_count} passed")

    return results

# Run the evaluation
results = await run_evaluation()
print(results)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Running experiment: 100%|██████████| 10/10 [03:24<00:00, 20.47s/it]

Results: 9/10 passed
Experiment(name=20260529-180933_myrag,  len=10)
